In [1]:
import numpy as np
import spacy
import re
import html
from collections import Counter
from mitreattack.stix20 import MitreAttackData


In [2]:
attack_data = MitreAttackData("/Users/jaytlinaskew/Downloads/enterprise-attack.json")  # Download from MITRE cti repo
techniques = attack_data.get_techniques()

for t in techniques:
    print(t['external_references'][0]['external_id'], "-", t['name'])
    print("Description:", t.get("description", "No description"))

T1055.011 - Extra Window Memory Injection
Description: Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data).(Citation: Microsoft Window Classes) Registration of new windows classes can include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data specific to that window and has specific application programming interface (API) functions to set and get its value. (Citation: Microsoft GetWindowLong function) (Citation: Microsoft SetWindowLong function)

Although small, t

In [3]:
def preprocess_description(text):
    if not text:
        return ""
    
    # Remove HTML tags and decode HTML entities
    text = re.sub(r'<.*?>', '', text)
    text = html.unescape(text)

    # Remove markdown-style links
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Remove inline code formatting (backticks)
    text = re.sub(r'`([^`]+)`', r'\1', text)

    # Remove (Citation: ...) blocks
    text = re.sub(r'\(Citation:.*?\)', '', text)
    
    # Remove the word "Adversaries" and similar filler terms
    filler_words = ['adversaries', 'may', 'possibly', 'can', 'could', 'might']
    for word in filler_words:
        text = re.sub(rf'\b{word}\b', '', text, flags=re.IGNORECASE)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


In [4]:
def extract_keywords(doc, top_n=10):
    # Filter out stop words, punctuation, short words
    tokens = [token.text.lower() for token in doc
              if token.is_alpha and not token.is_stop and len(token.text) > 2]
    
    # Get most common words
    common = Counter(tokens).most_common(top_n)
    return [word for word, count in common]

In [5]:
nlp = spacy.load("en_core_web_trf")
mitre_nlp_dict = {}

for tech in techniques:
    external_refs = tech.get("external_references", [])
    external_id = None
    for ref in external_refs:
        if ref.get("external_id", "").startswith("T"):
            external_id = ref["external_id"]
            break

    if not external_id:
        continue

    desc = tech.get("description", "").strip()
    if not desc:
        continue

    # Preprocess description
    desc = preprocess_description(desc)
    doc = nlp(desc)

    # Extract keywords and add to enhanced text
    keywords = extract_keywords(doc)
    enriched_text = desc + " " + " ".join(keywords)
    enriched_doc = nlp(enriched_text)

    name = tech.get("name", "Unnamed Technique")
    tactics = tech.get("kill_chain_phases", [])
    tactic_names = list(set(phase["phase_name"] for phase in tactics)) if tactics else []

    mitre_nlp_dict[external_id] = {
        "name": name,
        "description": desc,
        "tactics": tactic_names,
        "keywords": keywords,
        "spacy_doc": nlp(desc)
    }

In [6]:
mitre_nlp_dict

{'T1055.011': {'name': 'Extra Window Memory Injection',
  'description': "inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data). Registration of new windows classes include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data specific to that window and has specific application programming interface (API) functions to set and get its value. Although small, the EWM is large enough to store a 32-bit pointer and is often used to point to a windows procedure. Malware utilize this memory location in

In [7]:
print("Total techniques retrieved:", len(techniques))

Total techniques retrieved: 799


In [8]:
unique_techniques = set()
unique_tactics = set()

for v in mitre_nlp_dict.values():
    unique_techniques.add(v["name"])
    for tactic in v["tactics"]:
        unique_tactics.add(tactic)

print("\nUnique Techniques Loaded:", len(unique_techniques))
print("Sample Techniques:", list(unique_techniques)[:5])

print("\nUnique Tactics Loaded:", len(unique_tactics))
print("Sample Tactics:", list(unique_tactics))



Unique Techniques Loaded: 671
Sample Techniques: ['Source', 'Software Packing', 'Gather Victim Identity Information', 'Command and Scripting Interpreter', 'Masquerading']

Unique Tactics Loaded: 14
Sample Tactics: ['execution', 'defense-evasion', 'lateral-movement', 'resource-development', 'reconnaissance', 'impact', 'initial-access', 'command-and-control', 'discovery', 'credential-access', 'exfiltration', 'collection', 'persistence', 'privilege-escalation']


In [9]:
def get_vector(doc):
    # Get Ragged tensor from the transformer layer
    ragged = doc._.trf_data.last_hidden_layer_state
    
    tensor = ragged.data 

    if tensor.shape[0] == 0:
        return np.zeros((tensor.shape[1],))  # fallback zero vector

    return tensor.mean(0)

def infer_tactic_hints(text):
    hints = []
    if "network" in text or "traffic" in text:
        hints.append("command-and-control")
    if "initially access" in text or "entry point" in text:
        hints.append("initial-access")
    return hints

def semantic_tagging(text, dictionary, threshold=0.70, top_n=5, keyword_boost=0.1):
    input_doc = nlp(text)
    input_vec = get_vector(input_doc)
    text_lower = text.lower()

    matches = []

    for external_id, data in dictionary.items():
        candidate_vec = get_vector(data["spacy_doc"])

        if not np.any(input_vec) or not np.any(candidate_vec):
            continue

        sim = float(np.dot(input_vec, candidate_vec) /
                    (np.linalg.norm(input_vec) * np.linalg.norm(candidate_vec)))
        if np.isnan(sim):
            continue

        for keyword in data.get("keywords", []):
            if keyword.lower() in text_lower:
                sim += keyword_boost
                break
        
        keyword_match = sum(1 for k in data.get("keywords", []) if k in text_lower)
        tactic_match = sum(1 for t in data.get("tactics", []) if t in infer_tactic_hints(text_lower))

        final_score = round(sim + (keyword_boost * keyword_match) + (0.05 * tactic_match), 2)
    
        if sim >= threshold:
            matches.append({
                "technique_name": data["name"],
                "external_id": external_id,
                "tactics": data["tactics"],
                "similarity": round(sim, 3),
                "final_score": final_score
            })
    
    matches.sort(key=lambda x: x["final_score"], reverse=True)
    return matches[:1]


In [10]:
def preprocess_text(text):
    doc = nlp(text)
    lemmatized_tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and token.is_alpha
    ]
    return " ".join(lemmatized_tokens)

In [11]:
# Original unprocessed input
text = """
CISA conducted a hunt on IoC's obtained from ongoing investigations regarding recently disclosed CVE-2025-282 and CVE-2025-283.
CISA Analysts discovered traffic on 6JAN2025 between 198.98.54[.]209 and NLM device 130.14.13.10 that has similar traffic patterns to previously confirmed compromises at separate agencies.
CISA recommends NLM verifies their Ivanti versions and investigate this activity to determine if any malicious events have occurred.
"""
results = semantic_tagging(text, mitre_nlp_dict, threshold=0.9)

# Display results with final score
for match in results:
    print(f"✅ {match['external_id']} - {match['technique_name']}")
    print(f"  - Tactics: {', '.join(match['tactics'])}")
    print(f"  - Semantic Similarity: {match['similarity']}")
    print(f"  - Final Score: {match['final_score']}")


✅ T1562.006 - Indicator Blocking
  - Tactics: defense-evasion
  - Semantic Similarity: 0.97
  - Final Score: 1.27
